# 07 — Organizational Network & Span of Control
**Goal**: Analyze reporting structure, org design, and network centrality

**ML Progression**: Network construction → Centrality metrics → Span-of-control analysis

**HR Value**: Org design optimization, manager effectiveness

**Employee Value**: Understanding team structure and reporting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from pathlib import Path
import networkx as nx

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

cwd = Path.cwd()
if (cwd / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Cannot find project root')
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()

ANALYSIS_DIR = PROJECT_ROOT / 'data/analysis/07_network'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ANALYSIS_DIR / 'dataset.parquet')
print(f'Loaded: {len(df)} employees, {len(df.columns)} cols')

## 1. Network Construction

In [ ]:
G = nx.DiGraph()

for _, row in df.iterrows():
    emp = str(row['EmpID'])
    sup = str(row['Supervisor'])
    G.add_edge(sup, emp)

print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Connected components: {nx.number_weakly_connected_components(G)}')

## 2. Centrality Analysis

In [ ]:
betweenness = nx.betweenness_centrality(G)
closeness = nx.closeness_centrality(G)

top_between = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]
print('Top 10 by Betweenness Centrality:')
for node, score in top_between:
    print(f'  {node}: {score:.4f}')

In [ ]:
centrality_df = pd.DataFrame({
    'node': list(betweenness.keys()),
    'betweenness': list(betweenness.values()),
    'closeness': [closeness.get(n, 0) for n in betweenness.keys()]
}).sort_values('betweenness', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(centrality_df['betweenness'], bins=50, alpha=0.7)
ax.set_title('Betweenness Centrality Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/07_betweenness_dist.png', bbox_inches='tight')
plt.show()

## 3. Span of Control

In [ ]:
span = {node: G.out_degree(node) for node in G.nodes()}
span_df = pd.DataFrame({
    'manager': list(span.keys()),
    'direct_reports': list(span.values())
}).sort_values('direct_reports', ascending=False)

print('Span of Control:')
print(f'  Mean: {span_df["direct_reports"].mean():.1f}')
print(f'  Median: {span_df["direct_reports"].median():.1f}')
print(f'  Max: {span_df["direct_reports"].max()}')
print(f'  Min: {span_df["direct_reports"].min()}')
print(f'  Std: {span_df["direct_reports"].std():.1f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
span_df['direct_reports'].hist(bins=30, ax=ax, alpha=0.7)
ax.set_title('Distribution of Span of Control')
ax.set_xlabel('Direct Reports')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/07_span_of_control.png', bbox_inches='tight')
plt.show()

## 4. Key Takeaways

In [ ]:
print('--- Key Insights ---')
print(f'1. Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'2. Avg span of control: {span_df["direct_reports"].mean():.1f}')
print(f'3. Centrality reveals key structural positions')
print()
print('--- HR Action Items ---')
print('- Review managers with very wide span of control')
print('- Identify structural bottlenecks via centrality')
print()
print('--- Employee Impact ---')
print('- Balanced spans improve manager-employee relationships')
print('- Clear reporting structure supports career navigation')